# Advanced FileSet Features

This notebook demonstrates advanced FileSet capabilities:
- **Metadata filtering** — restrict which documents are used as seeds
- **RAG context generation** — retrieve supporting context with temporal constraints
- **RAG labeling** — resolve questions by searching the FileSet
- **Full combined pipeline** — context + labeling in one run

**Prerequisite**: Run `01_create_fileset.ipynb` first to create a FileSet and upload documents.

In [1]:
%pip install ../.. python-dotenv pandas -q

from IPython.display import clear_output
# clear_output()


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /Users/kskotheim/Desktop/dev/forecasting/lightningrod-python-sdk/.venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/sign-up?redirect=/api) to get your API key and **$50 of free credits**.

- **Google Colab**: Go to the Secrets section (key icon in left sidebar) and add a secret named `LIGHTNINGROD_API_KEY`
- **Local Jupyter**: Set the `LIGHTNINGROD_API_KEY` environment variable, or you'll be prompted to enter it

In [2]:
from dotenv import load_dotenv
from lightningrod import LightningRod
from lightningrod.utils import config

load_dotenv()
api_key = config.get_config_value("LIGHTNINGROD_API_KEY")

lr = LightningRod(api_key=api_key)

## Configure FileSet ID

Paste the FileSet ID from notebook 1 below.

In [3]:
fileset_id = "f049b2df-8622-4eac-9fcb-e8f51ef47d4f"

In [ ]:
import pandas as pd
from lightningrod import (
    QuestionPipeline,
    FileSetSeedGenerator,
    QdrantContextGenerator,
    QdrantRAGLabeler,
    QuestionGenerator,
    BinaryAnswerType,
    TemporalConstraint,
)

answer_type = BinaryAnswerType()

## Metadata Filtering

Use `metadata_filters` on the seed generator to restrict which files become seeds. Here we generate questions only from **APEX** documents.

In [6]:
pipeline_filtered = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='APEX'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions="Generate yes/no questions about the specific financial metrics and business events in these quarterly reports.",
        questions_per_seed=2,
    ),
)

dataset_filtered = lr.transforms.run(
    pipeline_filtered,
    name="FileSet - APEX Only (Metadata Filter)",
)
print(f"Dataset: {dataset_filtered.id}")
print(f"Rows: {dataset_filtered.num_rows}")

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  >> Pipeline Completed                                                                                          │
│                                                                                                                 │
│    Job ID:           3164f395-4f1b-4fd1-9fbd-752920e73e75                                                       │
│                                                                                                                 │
│    Total cost: $0.00                                                                                            │
│                                                                                                                 │
│  ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓  │
│  ┃ Step                              ┃ Progress                ┃  In ┃  Out ┃  Rejected ┃  Errors ┃ Duration ┃  │
│  ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩  │
│  │ FileSetSeedGeneratorTransform     │ Complete                │   5 │    5 │         0 │       0 │       1s │  │
│  │ QuestionGeneratorTransform        │ Complete                │   5 │   10 │         0 │       0 │       0s │  │
│  └───────────────────────────────────┴─────────────────────────┴─────┴──────┴───────────┴─────────┴──────────┘  │
│                                                                                                                 │
│    View full details:                                                                                           │
│  ]8;id=794685;https://dashboard.lightningrod.ai/?redirect=/datasets/25d93046-61a8-43ff-80fc-6ee6b3e62b5a\https://dashboard.lightningrod.ai/?redirect=/datasets/25d93046-61a8-43ff-80fc-6ee6b3e62b5a]8;;\                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Dataset: 25d93046-61a8-43ff-80fc-6ee6b3e62b5a
Rows: 10


In [7]:
samples_filtered = dataset_filtered.download()
for i, s in enumerate(samples_filtered[:3]):
    print(f"--- Sample {i+1} ---")
    print(f"Seed (first 120 chars): {s.seed.seed_text[:120]}...")
    print(f"Question: {s.question.question_text}")
    print()

--- Sample 1 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q1 2024
Period ending March 31, 2024

Financial Highlights:
- Revenu...
Question: Will APEX Technologies Inc. report total revenue of at least $2.20 billion in its quarterly financial results for the period ending June 30, 2024?

--- Sample 2 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q1 2025
Period ending March 31, 2025

Financial Highlights:
- Revenu...
Question: Will APEX Technologies Inc. announce the general availability of AXCompute 4.0 by the end of Q3 2025?

--- Sample 3 ---
Seed (first 120 chars): APEX Technologies Inc. — Quarterly Investor Report, Q4 2024
Period ending December 31, 2024

Financial Highlights:
- Rev...
Question: Will APEX Technologies Inc. report total revenue of at least $10.5 billion for the full fiscal year 2025?



## RAG Context Generation

`FileSetContextGenerator` retrieves supporting context from the FileSet for each generated question.

- **`metadata_filter_keys=["ticker"]`** — only retrieve context from the same company
- **`temporal_constraint=BEFORE`** — only retrieve context from documents dated before the seed, preventing lookahead bias

## RAG Context Generation

`QdrantContextGenerator` retrieves supporting context from the FileSet for each generated question.

- **`payload_filters={"ticker": "ticker"}`** — only retrieve context from the same company (maps Qdrant payload key to sample metadata key)
- **`temporal_constraint=TemporalConstraint.BEFORE`** — only retrieve context from documents dated before the seed, preventing lookahead bias

## RAG Labeling

`FileSetRAGLabeler` resolves questions by searching the FileSet for answers.

- **`temporal_constraint=AFTER`** — only search documents dated after the seed, so forward-looking questions are resolved by later reports
- **`confidence_threshold=0.7`** — only label questions where the labeler is at least 70% confident

## RAG Labeling

`QdrantRAGLabeler` resolves questions by searching the FileSet for answers.

- **`temporal_constraint=TemporalConstraint.AFTER`** — only search documents dated after the seed, so forward-looking questions are resolved by later reports
- **`confidence_threshold=0.7`** — only label questions where the labeler is at least 70% confident

In [ ]:
pipeline_labeler = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
        metadata_filters=["ticker='VGI'"],
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements, guidance, and planned initiatives "
            "mentioned in these quarterly reports. Focus on questions whose answers would be found in "
            "subsequent quarterly reports."
        ),
        questions_per_seed=2,
    ),
    labeler=QdrantRAGLabeler(
        file_set_id=fileset_id,
        payload_filters={"ticker": "ticker"},
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_labeler = lr.transforms.run(
    pipeline_labeler,
    max_seeds=3,  # Increase to ~5000 for a real run
    name="FileSet - RAG Labeler (VGI, AFTER)",
)
print(f"Dataset: {dataset_labeler.id}")
print(f"Rows: {dataset_labeler.num_rows}")

## Full Pipeline — Context + Labeling

Combine context generation and labeling in a single pipeline:
- **Context** (`BEFORE`) — retrieve earlier reports as supporting context
- **Labeler** (`AFTER`) — resolve forward-looking questions using later reports

In [ ]:
pipeline_full = QuestionPipeline(
    seed_generator=FileSetSeedGenerator(
        file_set_id=fileset_id,
        chunk_size=2000,
        chunk_overlap=200,
    ),
    question_generator=QuestionGenerator(
        answer_type=answer_type,
        instructions=(
            "Generate yes/no questions about forward-looking statements and guidance in these investor reports. "
            "Focus on questions that can be verified by looking at later quarterly reports for the same company."
        ),
        questions_per_seed=1,
    ),
    context_generators=[
        QdrantContextGenerator(
            file_set_id=fileset_id,
            payload_filters={"ticker": "ticker"},
            temporal_constraint=TemporalConstraint.BEFORE,
        ),
    ],
    labeler=QdrantRAGLabeler(
        file_set_id=fileset_id,
        payload_filters={"ticker": "ticker"},
        temporal_constraint=TemporalConstraint.AFTER,
        confidence_threshold=0.7,
        answer_type=answer_type,
    ),
)

dataset_full = lr.transforms.run(
    pipeline_full,
    max_seeds=8,
    name="FileSet - Full Pipeline (Context + Labeler)",
)
print(f"Dataset: {dataset_full.id}")
print(f"Rows: {dataset_full.num_rows}")

In [10]:
rows = dataset_full.flattened()
df = pd.DataFrame(rows)

cols = ["question_text", "label", "label_confidence", "reasoning", "is_valid"]
df[[c for c in cols if c in df.columns]]

,question_text,label,label_confidence,reasoning,is_valid
0,Will APEX Technologies Inc. report total reven...,1,1.0,"According to the 'apex_q2_2024.txt' document, ...",True
